In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
#不太适合有独热编码的

In [25]:
class Node:
    __slots__=("feature","threshold","left",
               "right","value","n_samples","impurity")
    #可以节省性能牺牲灵活度
    def __init__(self):
        self.feature=None
        self.threshold=None
        self.left=None
        self.right=None
        self.value=None
        self.n_samples=0
        self.impurity=0.0
        


In [48]:
class CARTRegressor:
    def __init__(self,max_depth=None,min_samples_split=2,
                min_samples_leaf=1,min_impurity_decrease=0.0):
        """min_samples_lef:分裂后的节点是否满足最小值要求
           min_samples_split：该节点是否满足分裂要求
        """
        self.max_depth=max_depth if max_depth is not None else np.iinfo(np.int64).max
        #如果没有设置那就无穷大
        self.min_samples_split=min_samples_split
        self.min_samples_leaf=min_samples_leaf
        self.min_impurity_decrease=min_impurity_decrease

    def fit(self,X,y):
        X=np.asarray(X,dtype=np.float64)
        y=np.asarray(y,dtype=np.float64).ravel()
        #y值变为一维
        self.n_features_=X.shape[1]#特征数
        self._N=X.shape[0]#总样本数
        self.feature_importances_=np.zeros(self.n_features_)
        self.root=self._build(X,y,depth=0)
        total=self.feature_importances_.sum()#归一化特征
        if total>0:
            self.feature_importances_/=total
        return self
        
    def _mse(self,y):
        return((y-y.mean())**2).mean()
        
    def _best_split(self,X,y):
        """返回(best_feature,best_threshold,子节点see之和)"""
        n=y.shape[0]
        total=y.sum()
        total_sq=(y*y).sum()
        #SSE简化
        best_loss=total_sq-total*total/n
        best_feature,best_threshold=None,None
        min_leaf=self.min_samples_leaf

        for j in range(self.n_features_):
            order=np.argsort(X[:,j],kind='mergesort')
            #order返回的是一个带着索引的列表
            xj=X[order,j]
            yj=y[order]
            #y的前缀和比如y=[10,20,30],那么csum=[10,30,60],csum_sq=[100,500,1400]
            csum=np.cumsum(yj)
            csum_sq=np.cumsum(yj*yj)
            #切分位置i左侧前i,又有最少样本数限制
            for i in range(min_leaf,n-min_leaf+1):
                #相同特征值跳过
                if xj[i-1]==xj[i]:
                    continue
                nL,nR=i,n-i
                sumL=csum[i-1]
                sumR=total-sumL
                sqL=csum_sq[i-1]
                sqR=total_sq-sqL
                #sse=平方和-和的平方/n
                sse_L=sqL-sumL*sumL/nL
                sse_R=sqR-sumR*sumR/nR
                loss=sse_L+sse_R
                if loss<best_loss-1e-12:
                    best_loss=loss
                    best_feature=j
                    best_threshold=0.5*(xj[i-1]+xj[i])
        return best_feature,best_threshold,best_loss
            
            
    
    def _build(self,X,y,depth):
        node=Node()
        node.n_samples=y.shape[0]
        node.value=y.mean()
        node.impurity=self._mse(y)

        if(depth>=self.max_depth or 
          node.n_samples<=self.min_samples_split or
          node.impurity<=1e-12):
            return node

        feature,threshold,sse_children=self._best_split(X,y)
        if feature is None:
            return node
        parent_sse=node.impurity*node.n_samples
        decrease=(parent_sse-sse_children)/self._N
        if decrease<self.min_impurity_decrease:
            return node

        mask=X[:,feature]<=threshold
        node.feature=feature
        node.threshold=threshold
        self.feature_importances_[feature]+=(parent_sse-sse_children)
        node.left=self._build(X[mask],y[mask],depth+1)
        node.right=self._build(X[~mask],y[~mask],depth+1)
        return node
    def predict(self,X):
        X=np.asarray(X,dtype=np.float64)
        return np.array([self._predict_one(row,self.root) for row in X])
        #记得带中括号否则返回生成器
    def _predict_one(self,x,node):
        while node.feature is not None:
            node=node.left if x[node.feature]<=node.threshold else node.right
        return node.value
    def get_depth(self,node=None):
        node=self.root_ if node is not None else node
        if node.feature is None:
            return 0
        return 1+max(self.get_depth(node.left),self.get_depth(node.right))

    def print_tree(self,feature_names=None,node=None,depth=0):
        node = self.root_ if node is None else node
        indent = "    " * depth
        if node.feature is None:
            print(f"{indent}└─ 预测 = {node.value:.4f}  (n={node.n_samples})")
            return
        name = feature_names[node.feature] if feature_names else f"X[{node.feature}]"
        print(f"{indent}├─ {name} <= {node.threshold:.4f} ?"
              f"   (mse={node.impurity:.4f}, n={node.n_samples})")
        self.print_tree(feature_names, node.left, depth + 1)
        self.print_tree(feature_names, node.right, depth + 1)


In [49]:
df=pd.read_csv("cart_regression.csv")
df.head()

,Square_Meters,House_Age,Distance_to_Center,Num_Bedrooms,House_Price
0,85,5,2.5,2,455.2
1,120,10,5.1,3,520.8
2,155,2,1.2,4,850.5
3,65,15,8.5,1,210.3
4,200,8,12.0,4,480.0


In [52]:
X=df.drop(columns='House_Price').values
y=df['House_Price'].values
House_Price=CARTRegressor()
House_Price.fit(X,y)
y_pred=House_Price.predict(X)
err=y-y_pred
(err**2).mean()

24.248599999999993